# Análise de Sintenia Colinear Intergenômica com JCVI (MCscan)

Este notebook descreve o pipeline para identificar, refinar e visualizar blocos de sintenia colinear entre os genomas de *V. peripherica* (*Vper*) e *V. tubiflora* (*Vtub*) utilizando a ferramenta **JCVI (MCscan em Python)**.

**Nova Organização Modular:**
Para evitar colisões e redundâncias, a análise intergenômica é realizada dentro do diretório dedicado `intergenomic/`:
- `intergenomic/intermediates/` para salvar arquivos intermediários (`.last`, `.anchors`, `.simple`).
- `intergenomic/layouts/` para as configurações dos plots (`seqids`, `layout`, `.blocks`).
- `intergenomic/results/` para as imagens geradas de macrosintenia.

**Pré-requisitos:**
- O ambiente conda `jcviEnv` deve estar ativo.
- Os links simbólicos globais devem ser gerados em `data/` rodando o script `scripts/prepare_files.py`.

## 1. Configurando o Ambiente de Execução

Navegamos para o diretório isolado de sintenia intergenômica antes de rodar os comandos.

In [ ]:
import os
# Garante que o notebook mude para o diretório de execução intergenômico
notebook_dir = os.getcwd()
target_dir = os.path.abspath(os.path.join(notebook_dir, "..", "intergenomic"))
os.chdir(target_dir)
print(f"Diretório de execução atual: {os.getcwd()}")

## 2. Busca de Ortólogos e Alinhamento Sintênico (Vper vs Vtub)

Executamos o alinhamento LAST. Para manter o diretório limpo de bases binárias pesadas de indexação, direcionamos o banco de dados do LAST para a pasta compartilhada `last_db/` usando a flag `--path=../last_db`.

In [ ]:
# Rodar o cálculo de sintenia cruzada direcionando indexação do LAST para last_db
!python -m jcvi.compara.synteny screen Vper.bed Vtub.bed --last --path=../last_db

Após a execução, os arquivos `.last`, `.anchors` e `.simple` serão gerados no diretório atual. 
Para manter a pasta limpa, as saídas são salvas em `intermediates/`.

In [ ]:
# Mover arquivos intermediários gerados para a pasta intermediates/
import glob
import shutil
for f in glob.glob("Vper.Vtub.*"): 
    if not os.path.isdir(f):
        shutil.move(f, os.path.join("intermediates", f))
print("Arquivos intermediários organizados com sucesso em intermediates/!")

## 3. Visualização de Macrosintenia (Karyotype Plot)

Para gerar a representação gráfica da macrosintenia entre os cromossomos das duas espécies, utilizamos os arquivos de layout estruturados em `layouts/`:
- `layouts/seqids`: Define os cromossomos a serem plotados (`vp_chr1A..9A` e `vt_chr1A..9A`).
- `layouts/layout`: Configura as posições dos eixos, cores modernas (azul claro e coral) e o link para o arquivo de âncoras filtrado.

In [ ]:
# Gerar o plot de Karyotype de macrosintenia cruzada
# Nota: Sempre use a flag '--notex' para evitar problemas locais de renderização do LaTeX
!python -m jcvi.graphics.karyotype layouts/seqids layouts/layout --notex

O arquivo final gerado será salvo no diretório isolado de resultados em `results/macrosynteny.pdf`.

## 4. Visualização de Microsintenia Focada (Micro-Synteny)

Para investigar locus gênicos locais (como os genes da família **OSCA** de forma focada e limpa):
Podemos rodar o pipeline automatizado que extrai as janelas gênicas locais (31 genes ao redor de cada gene OSCA) e plota os PDFs diretamente na pasta de resultados consolidados do projeto:

In [ ]:
# Voltar para a raiz das análises de sintenia e rodar o script de extração automática de OSCA
os.chdir("..")
!python scripts/extract_all_osca_microsynteny.py

Todos os plots locais gerados serão salvos em `results/microsynteny_plots/` com as anotações e realces de cores adequados.